# <span style="color:red">  Daily Data - EBC creation, FMB Quantiles (2000-2024)

In [1]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [2]:
from pathlib import Path
project_root = Path.cwd().parent
sys.path.append(str(project_root))

PROJECT_ROOT = Path.cwd().resolve().parents[0]
DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"


In [3]:
import importlib
import src.preference_factors.build_datasets as bd
import src.FamaMacBeth.betas as fmb_betas
import src.FamaMacBeth.pipeline as fmb_pipe
import src.FamaMacBeth.quantile as fmb_quantile
import src.FamaMacBeth.cross_sectional_regression as fmb_csr
import src.FamaMacBeth.pricing as fmb_pricing
import src.FamaMacBeth.backtest as fmb_backtest
import src.FamaMacBeth.plots as fmb_plots

# Reload the modules themselves
importlib.reload(bd)
importlib.reload(fmb_betas)
importlib.reload(fmb_pipe)
importlib.reload(fmb_quantile)
importlib.reload(fmb_csr)
importlib.reload(fmb_pricing)
importlib.reload(fmb_backtest)
importlib.reload(fmb_plots)


<module 'src.FamaMacBeth.plots' from '/Users/evi/Desktop/Invesco_Research_Project/src/FamaMacBeth/plots.py'>

# <span style="color:red"> Create CSV's for Daily EBC Weights, Beta Contributions, EBC returns, Preference Returns, Full preference and factors returns (Excess returns) </span>

In [ ]:
merged_df, full_merged_df = bd.build_all_dataset(
    returns_file="sp500_returns_daily_with_tickers.csv",
    market_caps_file="sp500_market_caps_daily.csv",
    ff_factors_file="ff_factors_daily.csv",
    frequency="daily"
)

ebc = merged_df["EBC"]
cw_ebc = merged_df["CW-EBC"]


Rolling OLS CAPM:   0%|          | 0/15229 [00:00<?, ?it/s]

EBC optimization (parallel):   0%|          | 0/15229 [00:00<?, ?it/s]

In [ ]:
np.log1p(merged_df[['CW', 'EW', 'EBC']]).cumsum().plot(figsize=(10,6))
plt.title("Cumulative Log Excess Returns")
plt.show()

# <span style="color:red">  Get rolling betas for the two focus portfolios EBC and CW - EBC

#### create factor returns

In [7]:
ebc = ebc.to_frame(name="EBC")
cw_ebc = cw_ebc.to_frame(name="CW-EBC")
factor_returns =  pd.concat([ebc, cw_ebc], axis=1).dropna()
factor_returns.columns = ["EBC", "CW-EBC"]

#### get asset returns and make them excess

In [8]:
asset_returns = bd.build_returns_dataset("sp500_returns_daily_with_tickers.csv", frequency="daily")
ff_factors = bd.build_ff_dataset("ff_factors_daily.csv", frequency="daily")

asset_returns, ff_factors = asset_returns.align(ff_factors, join="inner", axis=0)
# Create the Excess Returns for the Assets
asset_excess_returns = asset_returns.sub(ff_factors["RF"], axis=0)

#### compute rolling betas for EBC and CW-EBC

In [ ]:
asset_excess_returns, factor_returns = asset_excess_returns.align(factor_returns, join="inner", axis=0)

betas_ebc = fmb_betas.compute_rolling_betas(asset_excess_returns, ebc, rolling_window=252, min_obs=100)
betas_cw_ebc = fmb_betas.compute_rolling_betas(asset_excess_returns, cw_ebc, rolling_window=252, min_obs=100)


### CHECKS

In [ ]:
print("EBC betas shape:", betas_ebc.shape)
print("CW-EBC betas shape:", betas_cw_ebc.shape)

print("\nFirst 5 rows:")
print(betas_ebc.head())

In [ ]:
betas_ebc.stack().hist(bins=100)
plt.title("EBC Beta Distribution")
plt.show()

In [ ]:
betas_cw_ebc.stack().hist(bins=100)
plt.title("EBC Beta Distribution")
plt.show()

In [ ]:
# Count how many stocks have a valid beta each month
betas_ebc.count(axis=1).plot(title="Number of Stocks with Valid Betas Over Time")
plt.ylabel("Count")
plt.show()

In [ ]:
# Count how many stocks have a valid beta each month
betas_cw_ebc.count(axis=1).plot(title="Number of Stocks with Valid Betas Over Time")
plt.ylabel("Count")
plt.show()

# <span style="color:red">  Run full factor pipeline --> Two factor regressions FMB

In [72]:
start_date = "1999-12-01"

In [73]:
asset_excess_returns_trimmed = asset_excess_returns.loc[start_date:]
factor_returns_trimmed = factor_returns.loc[start_date:]
betas_ebc_trimmed = betas_ebc.loc[start_date:]
betas_cw_ebc_trimmed = betas_cw_ebc.loc[start_date:]

In [ ]:
results = fmb_pipe.run_full_factor_pipeline(
    beta1_df=betas_ebc_trimmed,
    beta2_df=betas_cw_ebc_trimmed,
    sp500=asset_excess_returns_trimmed,
    EBC=factor_returns_trimmed["EBC"],
    Cap_EBC=factor_returns_trimmed["CW-EBC"],
    n_q1=3,
    n_q2=3,
    min_assets_per_cell=3,
    f1_name="EBC",
    f2_name="CW-EBC",
    frequency = 'daily',
    formation_frequency='monthly'
)

results["mean_grid"]

In [ ]:
results["portrets_wide"][:5]

# <span style="color:red">  Backtest

In [76]:
# ============================
# USER PARAMETERS
# ============================
#start_date = "1970-01-01"

# ============================
# REQUIRED INPUTS FROM PIPELINE
# ============================
members_df = results["members"]
n_q1, n_q2 = results["mean_grid"].shape

# ============================
# LOAD RETURNS / CAPS / RF
# ============================
DATA_RAW_DIR = Path.cwd().parent / "data" / "raw"
sp_500, sp_caps, rf = fmb_backtest.load_daily_inputs(DATA_RAW_DIR)  # excess sp500

In [ ]:
# ============================
# RUN BACKTEST
# ============================
quantile_portfolios_returns = fmb_backtest.backtest_quantile_portfolios(
    members_df=members_df,
    returns_df=sp_500,
    caps_df=sp_caps,
    n_q1=n_q1,
    n_q2=n_q2,
    weight="cap",
    frequency='daily'
)

In [ ]:
quantile_portfolios_returns

In [79]:
rename_map = {
    "Q1_Q1": "EBC_High_Pref_High",
    "Q1_Q2": "EBC_High_Pref_Mid",
    "Q1_Q3": "EBC_High_Pref_Low",
    "Q2_Q1": "EBC_Mid_Pref_High",
    "Q2_Q2": "EBC_Mid_Pref_Mid",
    "Q2_Q3": "EBC_Mid_Pref_Low",
    "Q3_Q1": "EBC_Low_Pref_High",
    "Q3_Q2": "EBC_Low_Pref_Mid",
    "Q3_Q3": "EBC_Low_Pref_Low",
}

quantile_portfolios_returns = quantile_portfolios_returns.rename(columns=rename_map)


# <span style="color:red"> PLOTS

In [80]:
complete_df = full_merged_df.join(
    quantile_portfolios_returns,
    how="inner"
)


In [81]:
selected_columns = ['CW','EBC_High_Pref_High', 'EBC_High_Pref_Mid',
       'EBC_High_Pref_Low', 'EBC_Mid_Pref_High', 'EBC_Mid_Pref_Mid',
       'EBC_Mid_Pref_Low', 'EBC_Low_Pref_High', 'EBC_Low_Pref_Mid',
       'EBC_Low_Pref_Low']

In [ ]:
# ============================
# PLOTS
# ============================
fmb_plots.plot_cum_log_returns(
    complete_df[selected_columns],
    title=f"Daily Cumulative Log Excess Returns ({n_q1}x{n_q2} Cap-Weighted)",
)

In [ ]:
# FM expected-return grid (annualized %)
implied_returns = fmb_plots.plot_expected_return_grid(results["pricing"], results["fm_table"], n_q1, n_q2,frequency='daily')

In [ ]:
actual_returns = fmb_plots.plot_mean_grid(results["mean_grid"], annualize=True, frequency='daily')

# <span style="color:red"> Save Files

In [ ]:

DATA_PROCESSED_DIR_KEN = PROJECT_ROOT / "data" / "processed" / "Ken"

period = start_date[:4] + '_2024'
period

## Save implied daily returns

In [86]:
implied_returns_renamed = implied_returns

implied_returns_renamed = implied_returns_renamed.rename(
    index={
        1: "High EBC",
        2: "Medium EBC",
        3: "Low EBC"
    },
    columns={
        1: "High Preference",
        2: "Medium Preference",
        3: "Low Preference"
    }
)

implied_returns_renamed.to_csv(DATA_PROCESSED_DIR_KEN/ f"daily_{period}_implied_returns.csv")

## Save actual daily returns


In [87]:
actual_returns_renamed = actual_returns

actual_returns_renamed = actual_returns_renamed.rename(
    index={
        1: "High EBC",
        2: "Medium EBC",
        3: "Low EBC"
    },
    columns={
        1: "High Preference",
        2: "Medium Preference",
        3: "Low Preference"
    }
)

actual_excel = actual_returns_renamed / 100

actual_excel.to_csv(DATA_PROCESSED_DIR_KEN/ f"daily_{period}_actual_returns.csv")



In [88]:
quantile_portfolios_returns_copy = quantile_portfolios_returns.copy()

save_dir = DATA_PROCESSED_DIR / 'daily'

quantile_portfolios_returns_copy.to_csv(save_dir / f"daily_{period}_quantile_returns_3x3.csv")

# <span style="color:red"> Extra Stats

In [ ]:
# 1. Annualized Sharpe Ratio (Daily)
# We use 252 for daily and 12 for monthly
daily_rets = quantile_portfolios_returns
sharpe = (daily_rets.mean() / daily_rets.std()) * np.sqrt(252)

# 2. Maximum Drawdown
# Calculate cumulative growth from $1
cum_rets = (1 + daily_rets).cumprod()
running_max = cum_rets.cummax()
drawdown = (cum_rets - running_max) / running_max
max_dd = drawdown.min()

performance = pd.DataFrame({
    "Ann. Return (%)": daily_rets.mean() * 252 * 100,
    "Ann. Volatility (%)": daily_rets.std() * np.sqrt(252) * 100,
    "Sharpe Ratio": sharpe,
    "Max Drawdown (%)": max_dd * 100
})

# Display sorted by Sharpe to see the most efficient portfolio
print("Portfolio Risk-Reward Summary:")
display(performance.sort_values("Sharpe Ratio", ascending=False).round(3))

In [ ]:
import matplotlib.pyplot as plt

# Define window (252 days * 3 years)
window = 252 * 3 

# 1. Rolling Annualized Returns
rolling_ann_ret = daily_rets.rolling(window=window).mean() * 252

# 2. Rolling Annualized Volatility
rolling_ann_vol = daily_rets.rolling(window=window).std() * np.sqrt(252)

# 3. Rolling Sharpe Ratio
rolling_sharpe = rolling_ann_ret / rolling_ann_vol

# --- Plotting ---
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 12), sharex=True)

# Plot Returns
rolling_ann_ret.plot(ax=ax1, alpha=0.8)
ax1.set_title(f'Rolling {window//252}-Year Annualized Returns', fontsize=14)
ax1.set_ylabel('Annualized Return')
ax1.grid(True, linestyle='--', alpha=0.6)
ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

# Plot Sharpe
rolling_sharpe.plot(ax=ax2, alpha=0.8)
ax2.set_title(f'Rolling {window//252}-Year Sharpe Ratio', fontsize=14)
ax2.set_ylabel('Sharpe Ratio')
ax2.axhline(0, color='black', lw=1) # Zero line for reference
ax2.grid(True, linestyle='--', alpha=0.6)
ax2.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.show()

In [93]:
from pathlib import Path
PROJECT_ROOT = Path.cwd().resolve().parents[0]
DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

In [ ]:
selected_columns

In [ ]:
complete_df[selected_columns].rolling(252).corr()['CW'].loc['1990-01-01':]

In [95]:
temp_df = complete_df[selected_columns].rolling(252).corr().loc[:,'EBC_High_Pref_Low'].reset_index()

In [ ]:
temp_df[temp_df['level_1']== 'CW'].set_index('date').plot()

In [ ]:
quantile_portfolios_returns.corr()

In [102]:
quantile_portfolios_returns.index.name = "date"

In [103]:
save_dir = DATA_PROCESSED_DIR / 'daily'

In [104]:
quantile_portfolios_returns.to_csv(save_dir / "daily_quantile_returns_3x3.csv")

# <span style="color:red">  Fitted GARCH

In [105]:
from arch import arch_model
import numpy as np
import matplotlib.pyplot as plt


In [106]:
def fit_garch_model(returns):
    """
    Fit GARCH(1,1) with student t errors
    """

    returns = returns.dropna()

    # Scale to percentage
    returns_scaled = returns * 100

    am = arch_model(
        returns_scaled,
        mean="Constant",
        vol="GARCH",
        p=1,
        q=1,
        dist="t"
    )

    res = am.fit(disp="off")

    return res


In [ ]:
complete_df.columns

In [111]:
portfolio_name = 'CW'

In [ ]:
portfolio = complete_df[portfolio_name]

garch_res = fit_garch_model(portfolio)

print(garch_res.summary())


cond_vol = garch_res.conditional_volatility / 100  # scale back

plt.figure(figsize=(12,4))
plt.plot(cond_vol)
plt.title(f" {portfolio_name} - Conditional Volatility (GARCH)")
plt.ylabel("Volatility")
plt.grid(True)
plt.show()

mean_return = portfolio.mean()

cond_sharpe = mean_return / cond_vol

plt.figure(figsize=(12,4))
plt.plot(cond_sharpe)
plt.axhline(0, linestyle="--")
plt.title(f"{portfolio_name} Conditional Sharpe Ratio")
plt.grid(True)
plt.show()





In [ ]:
portfolio = complete_df["EBC_High_Pref_Low"]

garch_res = fit_garch_model(portfolio)

print(garch_res.summary())


cond_vol = garch_res.conditional_volatility / 100  # scale back

plt.figure(figsize=(12,4))
plt.plot(cond_vol)
plt.title("Conditional Volatility (GARCH)")
plt.ylabel("Volatility")
plt.grid(True)
plt.show()

mean_return = portfolio.mean()

cond_sharpe = mean_return / cond_vol

plt.figure(figsize=(12,4))
plt.plot(cond_sharpe)
plt.axhline(0, linestyle="--")
plt.title("Conditional Sharpe Ratio")
plt.grid(True)
plt.show()



